# Scratch: candidate generation comparison pivot

Reads `results/candidate_generation_comparison_validation.csv`, selects one candidate-set size `M`, and pivots the long-format metrics into a model-by-metric table for quick comparison.

In [7]:
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 30)

CODE_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "run_all.py").is_file())
RESULTS = CODE_ROOT / "results"
CSV_PATH = RESULTS / "candidate_generation_comparison_validation.csv"

df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df):,} rows from {CSV_PATH}")
print("Available m values:", sorted(df["m"].unique()))

Loaded 416 rows from c:\Users\Yong Rui\BT4222-Group-6-\results\candidate_generation_comparison_validation.csv
Available m values: [np.int64(50), np.int64(100), np.int64(150), np.int64(200), np.int64(250), np.int64(300), np.int64(350), np.int64(400), np.int64(450), np.int64(500), np.int64(550), np.int64(600), np.int64(650), np.int64(700), np.int64(750)]


In [8]:
M = 500

subset = df[df["m"] == M]
if subset.empty:
    raise ValueError(f"No rows found for m={M}. Available: {sorted(df['m'].unique())}")

metric_order = [
    "recall_pooled",
    "recall_impression_mean",
    "hit_rate",
    "catalogue_coverage",
    "long_tail_recall",
    "cold_article_recall",
    "personalisation",
    "novelty_bits",
]

pivot = (
    subset.pivot_table(index="model", columns="metric", values="value")
    .reindex(columns=[m for m in metric_order if m in subset["metric"].unique()])
    .round(4)
)

print(f"Validation metrics for m={M} (models x metrics)")
pivot

Validation metrics for m=500 (models x metrics)


metric,recall_pooled,recall_impression_mean,hit_rate,catalogue_coverage,long_tail_recall,cold_article_recall,personalisation,novelty_bits
model,,,,,,,,
content_entity,0.0225,0.0232,0.0329,0.7844,0.0337,0.0351,0.9155,15.7099
content_text,0.0210,0.0206,0.0307,0.9841,0.0287,0.0205,0.9299,15.7083
content_text_entity,0.0297,0.0303,0.0431,0.9419,0.0398,0.0468,0.9254,15.7082
content_two_tower_mlp_bpr,0.0921,0.0903,0.1268,0.6116,0.0624,0.0526,0.9285,15.6712
content_two_tower_rolling_popularity_750_750,0.7971,0.8154,0.8642,0.5200,0.0623,0.0351,0.8494,14.5748
popularity_freshness_prior_hour_clicks,0.9054,0.9129,0.9375,0.0464,0.5431,0.0000,0.6269,14.0467


## Raw long-format rows for the selected `M`

Useful for checking numerators/denominators behind each value.

In [9]:
subset.sort_values(["metric", "model"])[["model", "metric", "value", "numerator", "denominator"]].reset_index(drop=True)

,model,metric,value,numerator,denominator
0,content_entity,catalogue_coverage,0.784425,3.729000e+04,47538.0
1,content_text,catalogue_coverage,0.984055,4.678000e+04,47538.0
2,content_text_entity,catalogue_coverage,0.941857,4.477400e+04,47538.0
3,content_two_tower_mlp_bpr,catalogue_coverage,0.611595,2.907400e+04,47538.0
4,content_two_tower_rolling_popularity_750_750,catalogue_coverage,0.520047,2.472200e+04,47538.0
5,popularity_freshness_prior_hour_clicks,catalogue_coverage,0.046405,2.206000e+03,47538.0
6,content_entity,cold_article_recall,0.035088,1.200000e+01,342.0
7,content_text,cold_article_recall,0.020468,7.000000e+00,342.0
8,content_text_entity,cold_article_recall,0.046784,1.600000e+01,342.0
9,content_two_tower_mlp_bpr,cold_article_recall,0.052632,1.800000e+01,342.0
